##EXERCISE 1


Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
генерирајте одговор за секое прашање од податочното множество CommonSenseQA.
Од базата на знаење GenericsKB изберете контекст од вкупно 5 документи за секое
прашање со семантичко пребарување и векторска репрезентација на документите
со моделот all-MiniLM-L6-v2.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q datasets evaluate sacrebleu bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util

In [4]:
from huggingface_hub import login
login()

In [5]:
model_id = "meta-llama/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [6]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [7]:
embedding_models = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "distilroberta": "sentence-transformers/all-distilroberta-v1"
}

embedder = SentenceTransformer(embedding_models["minilm"])
embedder

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
commonSenseQA = load_dataset("tau/commonsense_qa", split="validation[:200]")

In [ ]:
commonSenseQA

Dataset({
    features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
    num_rows: 200
})

In [ ]:
# list first five samples
for i in range(5):
    ex = commonSenseQA[i]
    print("Question:", ex["question"])
    print("Question concept:", ex["question_concept"])
    print("Choices:", list(zip(ex["choices"]["label"], ex["choices"]["text"])))
    print("Answer:", ex["answerKey"])
    print("=" * 50)

Question: A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?
Question concept: revolving door
Choices: [('A', 'bank'), ('B', 'library'), ('C', 'department store'), ('D', 'mall'), ('E', 'new york')]
Answer: A
Question: What do people aim to do at work?
Question concept: people
Choices: [('A', 'complete job'), ('B', 'learn from each other'), ('C', 'kill animals'), ('D', 'wear hats'), ('E', 'talk to each other')]
Answer: A
Question: Where would you find magazines along side many other printed works?
Question concept: magazines
Choices: [('A', 'doctor'), ('B', 'bookstore'), ('C', 'market'), ('D', 'train station'), ('E', 'mortuary')]
Answer: B
Question: Where are  you likely to find a hamburger?
Question concept: hamburger
Choices: [('A', 'fast food restaurant'), ('B', 'pizza'), ('C', 'ground up dead cows'), ('D', 'mouth'), ('E', 'cow carcus')]
Answer: A
Question: James was looking for a good place to buy farmland.  Where might he lo

In [8]:
def load_genericskb(limit=None):
    kb = load_dataset("generics_kb", split="train")
    if limit:
        kb = kb.select(range(limit))
    documents = [row["generic_sentence"] for row in kb]
    return documents

In [9]:
documents = load_genericskb(limit=None)

README.md: 0.00B [00:00, ?B/s]

generics_kb_best/train-00000-of-00001.pa(…):   0%|          | 0.00/39.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1020868 [00:00<?, ? examples/s]

In [10]:
def encode_texts(texts, model_name=embedding_models["minilm"]):
    embedder = SentenceTransformer(model_name)
    embeddings = embedder.encode(texts, convert_to_tensor=True)
    return model, embeddings

In [11]:
model, doc_embeddings = encode_texts(documents)

In [12]:
doc_embeddings[:5]

tensor([[ 0.0258,  0.0365, -0.0636,  ...,  0.0092, -0.0345, -0.0370],
        [-0.0111, -0.0863, -0.0271,  ..., -0.0550,  0.0290, -0.0216],
        [-0.0223,  0.1054, -0.0091,  ...,  0.0355, -0.0262,  0.0234],
        [ 0.0312,  0.0375, -0.0288,  ..., -0.0741,  0.0032,  0.0326],
        [-0.0533,  0.0858, -0.0216,  ...,  0.0032, -0.0085, -0.0023]],
       device='cuda:0')

In [13]:
def retrieve_top_k(question, documents, doc_embeddings, embedder, k=5):
    query_emb = embedder.encode([question], convert_to_tensor=True)
    cosine_scores = util.cos_sim(query_emb, doc_embeddings)[0]
    top_results = torch.topk(cosine_scores, k=k)
    top_docs = [documents[idx] for idx in top_results.indices]
    top_scores = [cosine_scores[idx].item() for idx in top_results.indices]

    return top_docs, top_scores

In [ ]:
def format_prompt(example, context=None):
    choices_text = "\n".join(
        f"{label}. {text}" for label, text in zip(example["choices"]["label"], example["choices"]["text"])
    )
    prompt = f"Question: {example['question']}\nChoices:\n{choices_text}\nAnswer (one letter):"
    if context:
        prompt = f"Context: {context}\n\n{prompt}"
    return prompt

In [ ]:
def generate_answer(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(text)
    if "Answer" in text:
        answer_part = text.split("Answer")[-1]
    else:
        answer_part = text
    for c in answer_part:
        if c in ["A", "B", "C", "D", "E"]:
            return c

    return "(not found)"

In [ ]:
def print_results(example, predicted, top_docs, top_scores=None):
    print("\n" + "="*80)
    print("QUESTION:", example["question"])
    print("Choices:", list(zip(example["choices"]["label"], example["choices"]["text"])))
    print("Gold Answer:", example["answerKey"])
    print("Predicted Answer:", predicted)
    print("\nTop documents retrieved as context:")
    for i, doc in enumerate(top_docs, 1):
        score_str = f" (score={top_scores[i-1]:.4f})" if top_scores is not None else ""
        print(f"{i}. {doc}{score_str}")
    print("=" * 90)

In [37]:
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score as bert_score
import numpy as np

In [ ]:
def evaluate(dataset, documents, doc_embeddings, embedder, tokenizer, model,
             k=5, use_rag=True, print_examples=False):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["question"], documents, doc_embeddings, embedder, k) if use_rag else []
        context = " ".join(top_docs) if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["answerKey"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

In [ ]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate(commonSenseQA, documents, doc_embeddings, embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

Streaming output truncated to the last 5000 lines.
C. pass water
D. listen to each other
E. sing
Answer (one letter): D. listen to each other

Explanation: Animals often communicate with each other through

QUESTION: What do animals do when an enemy is approaching?
Choices: [('A', 'feel pleasure'), ('B', 'procreate'), ('C', 'pass water'), ('D', 'listen to each other'), ('E', 'sing')]
Gold Answer: D
Predicted Answer: D

Top documents retrieved as context:
1. Animals have enemies. (score=0.7551)
2. Animals pursue prey. (score=0.7470)
3. Animals target prey. (score=0.7270)
4. Animals are attacked by predators. (score=0.7126)
5. Animals respond to threats in many complex ways. (score=0.7116)
Context: Reading helps thinking, writing and spelling. Reading cause knowledge. Reading helps individuals form their own opinions and strengthens their sense of self. Reading cause increase knowledge. Reading is used to stimulate writing.

Question: Reading newspaper one of many ways to practice your w

In [35]:
import evaluate as ev
bleu = ev.load("bleu")
bert = ev.load("bertscore")

In [ ]:
def letters_to_texts(preds, refs, dataset):
    pred_texts = []
    ref_texts = []
    for p, r, ex in zip(preds, refs, dataset):
        mapping = {label: text for label, text in zip(ex["choices"]["label"], ex["choices"]["text"])}
        pred_texts.append(mapping[p])
        ref_texts.append(mapping[r])
    return pred_texts, ref_texts

pred_texts, ref_texts = letters_to_texts(pred, refs, commonSenseQA)
bleu_score = bleu.compute(predictions=pred_texts, references=[[r] for r in ref_texts])
bert_score = np.mean(bert.compute(predictions=pred_texts, references=ref_texts, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

BLEU (text): {'bleu': 0.4762921106592682, 'precisions': [0.5353846153846153, 0.568, 0.4230769230769231, 0.4], 'brevity_penalty': 1.0, 'length_ratio': 1.025236593059937, 'translation_length': 325, 'reference_length': 317}
BERTScore F1 (text): 0.9469073575735092


The evaluation results indicate that the RAG approach with LLaMA-2 (4-bit quantized) performs very well on the CommonSenseQA task. The BLEU score of 47.6% shows that the generated answers have substantial n-gram overlap with the reference answers, which is particularly impressive given that many answers are short phrases or even single words. The breakdown of n-gram precisions demonstrates that the model consistently captures key words and partial sequences, while minor drops for 3-grams and 4-grams are expected due to the brevity of the answers. More importantly, the BERTScore F1 of 0.947 reflects a very high semantic similarity between the predicted and reference answers, indicating that the model reliably captures the intended meaning even when exact word matches are not perfect. Overall, these results suggest that combining RAG with LLaMA-2 and a semantic retrieval mechanism allows for accurate and semantically coherent question answering, making it a robust approach for short-answer QA tasks like CommonSenseQA.

Многу време и ресурси(GPU)(usage limits) е да го пробам и со другиот embedding model, меѓутоа значително подобри резултати се добиваат со дополнителниот контекст од документи кои се даваат на моделот за збогатување на знаењето.

## EXERCISE 2

Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
генерирајте одговор за секое прашање од податочното множество RAG-MiniWikipedia. Изберете контекст од вкупно 5 документи за секое прашање со
семантичко пребарување и векторска репрезентација на документите со моделот
all-MiniLM-L6-v2.

In [ ]:
wiki = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "question-answer",
    split="test[:200]"
)

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/test.parquet/part.0.parquet:   0%|          | 0.00/54.4k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

In [ ]:
wiki.column_names

['question', 'answer', 'id']

In [ ]:
# list first five samples
for i in range(5):
    ex = wiki[i]
    print("Question:", ex["question"])
    print("Answer:", ex["answer"])

Question: Was Abraham Lincoln the sixteenth President of the United States?
Answer: yes
Question: Did Lincoln sign the National Banking Act of 1863?
Answer: yes
Question: Did his mother die of pneumonia?
Answer: no
Question: How many long was Lincoln's formal education?
Answer: 18 months
Question: When did Lincoln begin his political career?
Answer: 1832


In [ ]:
def format_prompt(example, context=None):
    prompt = f"Question: {example['question']}\nAnswer:"
    if context:
        prompt = f"Context: {context}\n\n{prompt}"
    print(prompt)
    return prompt

def generate_answer(tokenizer, model, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Answer" in text:
        answer = text.split("Answer")[-1].strip()
    else:
        answer = text.strip()
    return answer

In [ ]:
def print_results(example, predicted, top_docs, top_scores):
    print("\n" + "="*80)
    print("QUESTION:", example["question"])
    print("Gold Answer:", example["answer"])
    print("Predicted Answer:", predicted)
    print("Top documents retrieved as context:")
    for i, (doc, score) in enumerate(zip(top_docs, top_scores), 1):
        print(f"{i}. ({score:.3f}) {doc}")
    print("\n" + "="*150)

In [ ]:
def evaluate(dataset, documents, doc_embeddings, embedder, tokenizer, model,
             k=5, use_rag=True, print_examples=False):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["question"], documents, doc_embeddings, embedder, k) if use_rag else []
        context = " ".join(top_docs) if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["answer"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

The model occasionally generates contradictory discourse markers (e.g., starting an answer with “No” while providing a factually correct explanation). This behavior stems from the autoregressive nature of large language models, which optimize for token-level likelihood rather than global logical consistency. To ensure reliable evaluation, answers were constrained and normalized during post-processing.

In [ ]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate(wiki, documents[:], doc_embeddings[:], embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

Evaluating with RAG (5 context docs)...
Context: Lincoln isa thing. Presidents to lead nations. Presidents are heads of state. Presidents are leaders. Presidents are used for leading.

Question: Was Abraham Lincoln the sixteenth President of the United States?
Answer:

QUESTION: Was Abraham Lincoln the sixteenth President of the United States?
Gold Answer: yes
Predicted Answer: : No, Abraham Lincoln was the sixteenth President of the United States.

Explanation: Abraham Lincoln was the 16th President of the United States, serving from 1861 until his assassination in 186
Top documents retrieved as context:
1. (0.508) Lincoln isa thing.
2. (0.503) Presidents to lead nations.
3. (0.481) Presidents are heads of state.
4. (0.478) Presidents are leaders.
5. (0.466) Presidents are used for leading.

Context: Lincoln isa thing. Proclamation is acts. A national bank is a bank Banking is conservatism. Banking is a noble profession.

Question: Did Lincoln sign the National Banking Act of 1863?
An

In [ ]:
bleu_score = bleu.compute(predictions=pred, references=[[r] for r in refs])
bert_score = np.mean(bert.compute(predictions=pred, references=refs, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU (text): {'bleu': 0.0062194324341350105, 'precisions': [0.04021813224267212, 0.0075672645739910315, 0.0033145986453379447, 0.0014832393948383269], 'brevity_penalty': 1.0, 'length_ratio': 8.450460829493087, 'translation_length': 7335, 'reference_length': 868}
BERTScore F1 (text): 0.8218814924359321


The results for the Wikipedia RAG task indicate that the model is not optimized for precise short-answer generation without downstream fine-tuning. The extremely low BLEU (0.6%) reflects that the predicted answers have almost no word overlap with references, likely due to verbose outputs. However, the BERTScore F1 of 0.82 shows that the model still captures the general semantic content of the answers. Overall, this suggests that LLaMA-2 with 4-bit quantization can produce semantically relevant but overly long or imprecise answers in a zero-shot setting. To improve results, the model would benefit from fine-tuning on RAG-style QA with Wikipedia passages or better prompt engineering to generate concise, precise answers.

##EXERCISE 3

Користејќи го моделот LLaMA-2 со квантизација со 4bits и техниката RAG,
трансформирајте ги речениците кои содржат негативен сентимент од податочното
множество Yelp_parallel во реченици со позитивен сентимент. Изберете контекст
од вкупно 5 документи за секое прашање со семантичко пребарување и векторска
репрезентација на документите со моделот all-MiniLM-L6-v2.

In [14]:
import pandas as pd

In [16]:
df = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt', sep='\t', header=None)
df.columns = ["Negative", "Positive"]
df = df.iloc[1:].reset_index(drop=True)
df.head(6)

,Negative,Positive
0,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
1,there is definitely not enough room in that pa...,There is so much room in that part of the venue
2,so basically tasted watered down.,It didn't taste watered down at all.
3,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."
4,i can't believe how inconsiderate this pharmac...,This pharmacy is really considerate.
5,just left and took it off the bill.,just left and put it on the bill.


In [19]:
yelp = load_dataset(
    "csv",
    data_files="/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt",
    delimiter="\t",
    split="train"
)
yelp

Dataset({
    features: ['Style 1', 'Style 2'],
    num_rows: 1000
})

In [18]:
print(yelp[0]["Style 1"])
print(yelp[0]["Style 2"])

ever since joes has changed hands it's just gotten worse and worse.
Ever since joes has changed hands it's gotten better and better.


In [29]:
def format_prompt(example, context=None):
    prompt = ""
    if context:
        prompt += "Context (positive examples):\n"
        for i, doc in enumerate(context, 1):
            prompt += f"{i}. {doc}\n"
        prompt += "\n"
    prompt += f"Negative review: {example['Style 1']}\n"
    prompt += "Rewrite it to be POSITIVE in **one sentence**:\n"

    return prompt

def generate_answer(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Rewrite it to be POSITIVE in **one sentence**:" in text:
        return text.split("Rewrite it to be POSITIVE in **one sentence**:")[-1].strip()
    return text.strip()

def print_results(example, predicted, top_docs, top_scores):
    print("\n" + "="*80)
    print("Negative:", example["Style 1"])
    print("Gold Positive:", example["Style 2"])
    print("Predicted Positive:", predicted)
    print("\nTop documents retrieved as context:")
    for i, (doc, score) in enumerate(zip(top_docs, top_scores), 1):
        print(f"{i}. ({score:.3f}) {doc}")

In [33]:
def evaluate_dataset(dataset, documents, doc_embeddings, embedder, tokenizer, model, k=5, use_rag=True, print_examples=True):
    refs, preds = [], []

    for ex in dataset:
        top_docs, top_scores = retrieve_top_k(ex["Style 1"], documents, doc_embeddings, embedder, k) if use_rag else ([], [])
        context = top_docs if top_docs else None
        prompt = format_prompt(ex, context)
        pred = generate_answer(tokenizer, model, prompt)
        refs.append(ex["Style 2"])
        preds.append(pred)

        if print_examples:
            print_results(ex, pred, top_docs, top_scores)

    return refs, preds

In [34]:
print("Evaluating with RAG (5 context docs)...")
refs, pred = evaluate_dataset(yelp, documents[:], doc_embeddings[:], embedder, tokenizer, model,
                             k=5, use_rag=True, print_examples=True)

Streaming output truncated to the last 5000 lines.
Negative: everything we've ever ordered her has been horrible tasting
Gold Positive: everything we've ever ordered here has been great tasting.
Predicted Positive: Everything we've ever ordered at this restaurant has been delicious and flavorful!

Top documents retrieved as context:
1. (0.529) Flesh has fine flavour.
2. (0.517) Good flavour is essential to consumer satisfaction.
3. (0.516) Tastings is eating.
4. (0.506) Flavor follows maternal consumption.
5. (0.497) Flavour has mild aroma.

Negative: it is the least authentic thai in the valley
Gold Positive: it is the most authentic thai in the valley.
Predicted Positive: It may not be the most authentic Thai food in the valley, but it still offers a unique and flavorful dining experience.

Top documents retrieved as context:
1. (0.636) Thai culture is a quirky mix of old and new, Western and Eastern.
2. (0.613) Thai people believe there's a spirit in their heads.
3. (0.606) Thai foo

In [38]:
bleu_score = bleu.compute(predictions=pred, references=[[r] for r in refs])
bert_score = np.mean(bert.compute(predictions=pred, references=refs, lang="en")['f1'])
print("BLEU (text):", bleu_score)
print("BERTScore F1 (text):", bert_score)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU (text): {'bleu': 0.04003995550917002, 'precisions': [0.14551765569721647, 0.05082346442012223, 0.025604348448004577, 0.01357315038012238], 'brevity_penalty': 1.0, 'length_ratio': 3.116820971600957, 'translation_length': 29962, 'reference_length': 9613}
BERTScore F1 (text): 0.8827826775312424


The results show that the RAG-LLaMA-2 model effectively transforms negative Yelp reviews into positive sentences. The BLEU score is low (0.04), reflecting minimal exact word overlap, mainly due to longer, more elaborate model outputs. However, the BERTScore F1 of 0.88 indicates strong semantic alignment with the references, demonstrating that the model successfully captures the intended positive sentiment. Overall, while BLEU underestimates performance, the approach reliably produces semantically correct positive rewrites, with room for improvement in brevity and style consistency.